# 03 - RAG Pipeline: Retrieval-Augmented Generation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/genai-arch/notebooks/03-rag-pipeline.ipynb)

## Learning Objectives

By the end of this notebook, you will be able to:

1. Split documents into chunks suitable for embedding
2. Generate embeddings using the OpenAI Embeddings API
3. Store and query embeddings using ChromaDB (a vector database)
4. Perform similarity search to retrieve relevant context
5. Build a complete RAG pipeline: retrieve, augment, generate
6. Evaluate retrieval quality with relevance scoring

---

**Architecture Pattern:** RAG lets an LLM answer questions about your own data by retrieving relevant documents and injecting them into the prompt. The model generates answers grounded in your content rather than relying solely on its training data.

## Setup

Install the required packages.

In [ ]:
!pip install openai chromadb tiktoken --quiet

## Configuration

In [ ]:
import os
import json
import hashlib
import numpy as np
from openai import OpenAI

api_key = os.environ.get("OPENAI_API_KEY", "")
if not api_key:
    print("WARNING: OPENAI_API_KEY not set. Mock responses will be used.")
    print("To use the real API, set your key: export OPENAI_API_KEY='sk-...'")
    print("ChromaDB operations will run locally (no API key needed).")
else:
    print("API key found. Using live OpenAI API.")

client = OpenAI(api_key=api_key if api_key else "sk-mock-key")
MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

---
## 1. Document Chunking

Before embedding, documents must be split into chunks. Chunking strategy significantly impacts retrieval quality:

- **Chunk size**: Too small means missing context; too large dilutes relevance
- **Overlap**: Overlapping chunks prevent information loss at boundaries
- **Boundaries**: Prefer natural splits (sentences, paragraphs) over arbitrary character counts

In [ ]:
# Sample knowledge base -- fictional company documentation
# These documents contain information the LLM was never trained on
DOCUMENTS = [
    {
        "id": "product-overview",
        "title": "AcmeCorp Product Overview",
        "content": (
            "AcmeCorp offers three main products: DataFlow Pro, InsightEngine, "
            "and CloudSync. DataFlow Pro is a real-time data pipeline tool that "
            "processes up to 10 million events per second. It supports Apache Kafka, "
            "Apache Pulsar, and Amazon Kinesis as source connectors. Pricing starts "
            "at $500/month for the Starter tier, $2,000/month for Business, and "
            "custom pricing for Enterprise. InsightEngine is an AI-powered analytics "
            "platform that generates dashboards and reports from natural language "
            "queries. It connects to PostgreSQL, MySQL, BigQuery, Snowflake, and "
            "Redshift. CloudSync provides file synchronization across multi-cloud "
            "environments (AWS, GCP, Azure) with end-to-end encryption and "
            "99.99% uptime SLA."
        ),
    },
    {
        "id": "dataflow-setup",
        "title": "DataFlow Pro Setup Guide",
        "content": (
            "To set up DataFlow Pro: 1) Install the CLI with `pip install dataflow-pro`. "
            "2) Run `dfp init --project my-project` to create a configuration file. "
            "3) Configure your source connector in `config.yaml` under the `sources` "
            "section. Supported sources are kafka, pulsar, and kinesis. 4) Define "
            "your transformation pipeline using the DataFlow DSL in `pipeline.dfp`. "
            "5) Deploy with `dfp deploy --env production`. The CLI supports "
            "hot-reloading: changes to pipeline.dfp are applied without downtime. "
            "For monitoring, use `dfp status` to see throughput, latency, and "
            "error rates. Logs are sent to the configured sink (CloudWatch, "
            "Cloud Logging, or a custom endpoint)."
        ),
    },
    {
        "id": "insightengine-features",
        "title": "InsightEngine Feature List",
        "content": (
            "InsightEngine key features: Natural Language Queries -- ask questions "
            "like 'show me revenue by region for Q3' and get auto-generated charts. "
            "Anomaly Detection -- ML-based alerts when metrics deviate from patterns. "
            "Scheduled Reports -- email PDF/CSV reports on daily, weekly, or monthly "
            "cadence. Data Governance -- role-based access control (RBAC) with "
            "column-level security. API Access -- REST and GraphQL APIs for "
            "embedding dashboards in your application. The platform uses a "
            "proprietary query optimizer called TurboQuery that is 3x faster "
            "than standard SQL engines on analytical workloads. InsightEngine "
            "supports SSO via SAML 2.0 and OIDC."
        ),
    },
    {
        "id": "pricing-faq",
        "title": "AcmeCorp Pricing FAQ",
        "content": (
            "Q: Is there a free trial? A: Yes, all products offer a 14-day free "
            "trial with full functionality. No credit card required. "
            "Q: Can I switch plans? A: Yes, you can upgrade or downgrade at any "
            "time. Changes take effect at the next billing cycle. "
            "Q: Do you offer annual discounts? A: Yes, annual billing gives a 20%% "
            "discount compared to monthly pricing. "
            "Q: What payment methods do you accept? A: We accept credit cards, "
            "ACH transfer, wire transfer, and purchase orders for Enterprise plans. "
            "Q: Is there a setup fee? A: No setup fees for Starter or Business "
            "plans. Enterprise plans include dedicated onboarding at no extra cost."
        ),
    },
    {
        "id": "security-compliance",
        "title": "Security and Compliance",
        "content": (
            "AcmeCorp is SOC 2 Type II certified and GDPR compliant. All data is "
            "encrypted at rest (AES-256) and in transit (TLS 1.3). We support "
            "customer-managed encryption keys (CMEK) for Enterprise plans. Our "
            "infrastructure runs on AWS and GCP with multi-region redundancy. "
            "We conduct annual penetration testing through independent third-party "
            "firms. Data residency options are available for EU, US, and APAC "
            "regions. All employee access requires MFA and is logged in an "
            "immutable audit trail."
        ),
    },
]

print(f"Knowledge base: {len(DOCUMENTS)} documents")
for doc in DOCUMENTS:
    print(f"  - {doc['title']} ({len(doc['content'])} chars)")

In [ ]:
def chunk_text(text, chunk_size=250, overlap=50):
    """Split text into overlapping chunks, preferring sentence boundaries."""
    if len(text) <= chunk_size:
        return [text]

    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]

        # Try to break at a sentence boundary
        if end < len(text):
            last_period = chunk.rfind(". ")
            if last_period > chunk_size // 2:
                chunk = chunk[:last_period + 1]
                end = start + last_period + 1

        chunks.append(chunk.strip())
        start = end - overlap

    return chunks


# Chunk all documents
all_chunks = []
all_ids = []
all_metadatas = []

for doc in DOCUMENTS:
    chunks = chunk_text(doc["content"], chunk_size=250, overlap=50)
    for i, chunk in enumerate(chunks):
        chunk_id = f"{doc['id']}_chunk_{i}"
        all_chunks.append(chunk)
        all_ids.append(chunk_id)
        all_metadatas.append({
            "source_doc": doc["id"],
            "title": doc["title"],
            "chunk_index": i,
        })

print(f"Created {len(all_chunks)} chunks from {len(DOCUMENTS)} documents:\n")
for i, chunk in enumerate(all_chunks):
    print(f"  [{all_ids[i]}] ({len(chunk)} chars) {chunk[:80]}...")

---
## 2. Embedding Generation

Embeddings convert text into dense vectors that capture semantic meaning. Similar texts produce vectors that are close together in the embedding space. We use OpenAI's `text-embedding-3-small` model (1536 dimensions).

In [ ]:
def generate_mock_embedding(text, dimensions=1536):
    """Generate a deterministic mock embedding from text hash."""
    seed = int(hashlib.md5(text.encode()).hexdigest()[:8], 16)
    rng = np.random.RandomState(seed)
    vec = rng.randn(dimensions).astype(np.float32)
    vec = vec / np.linalg.norm(vec)
    return vec.tolist()


def get_embedding(text, model=EMBEDDING_MODEL):
    """Get embedding for a text string. Falls back to mock."""
    try:
        response = client.embeddings.create(model=model, input=text)
        return response.data[0].embedding
    except Exception as e:
        print(f"API call failed: {e}")
        result = generate_mock_embedding(text)
        print("Using mock response for demonstration.")
        return result


def get_embeddings_batch(texts, model=EMBEDDING_MODEL):
    """Get embeddings for multiple texts in one API call."""
    try:
        response = client.embeddings.create(model=model, input=texts)
        return [item.embedding for item in response.data]
    except Exception as e:
        print(f"API call failed: {e}")
        result = [generate_mock_embedding(t) for t in texts]
        print("Using mock response for demonstration.")
        return result


# Demo: embed a single text
sample_embedding = get_embedding("What is DataFlow Pro?")
print(f"Embedding dimensions: {len(sample_embedding)}")
print(f"First 10 values: {[round(v, 4) for v in sample_embedding[:10]]}")
print(f"Vector norm: {np.linalg.norm(sample_embedding):.4f}")

In [ ]:
# Demonstrate semantic similarity
test_pairs = [
    ("What does DataFlow Pro cost?", "pricing for the data pipeline product"),
    ("What does DataFlow Pro cost?", "how to bake a chocolate cake"),
    ("Is AcmeCorp SOC 2 certified?", "security compliance and certifications"),
    ("Is AcmeCorp SOC 2 certified?", "best pizza restaurants in New York"),
]

print(f"{'Text A':<40} {'Text B':<42} {'Similarity':>10}")
print("-" * 95)

for text_a, text_b in test_pairs:
    emb_a = get_embedding(text_a)
    emb_b = get_embedding(text_b)
    similarity = float(np.dot(emb_a, emb_b))
    print(f"{text_a[:38]:<40} {text_b[:40]:<42} {similarity:>10.4f}")

---
## 3. Vector Store (ChromaDB)

ChromaDB is an open-source embedding database that handles storage, indexing, and similarity search. It runs locally with no API key needed. We create a collection, add our chunks with embeddings, and query it.

In [ ]:
import chromadb

# Create an in-memory ChromaDB client
chroma_client = chromadb.Client()

# Create a collection
collection = chroma_client.get_or_create_collection(
    name="acmecorp_docs",
    metadata={"description": "AcmeCorp product documentation"}
)

# Embed all chunks and add to the collection
print("Embedding and indexing chunks...")
chunk_embeddings = get_embeddings_batch(all_chunks)

collection.add(
    ids=all_ids,
    documents=all_chunks,
    embeddings=chunk_embeddings,
    metadatas=all_metadatas
)

print(f"Indexed {collection.count()} chunks in ChromaDB.")
print(f"\nChunk breakdown:")
for doc in DOCUMENTS:
    n = len([m for m in all_metadatas if m["source_doc"] == doc["id"]])
    print(f"  {doc['title']}: {n} chunks")

---
## 4. Similarity Search

Given a user query, we embed it and find the most similar chunks in our vector store. ChromaDB returns results ranked by distance (lower = more similar).

In [ ]:
def search(query, n_results=3):
    """Search the vector store for chunks most relevant to the query."""
    query_embedding = get_embedding(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    output = []
    for i in range(len(results["ids"][0])):
        output.append({
            "id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": round(results["distances"][0][i], 4),
        })
    return output


# Demo: search for relevant chunks
queries = [
    "How much does DataFlow Pro cost?",
    "How do I install and set up DataFlow?",
    "Is AcmeCorp SOC 2 certified?",
    "Does InsightEngine support natural language queries?",
]

for query in queries:
    print(f"\n=== Query: '{query}' ===")
    results = search(query, n_results=2)
    for j, r in enumerate(results):
        print(f"  [{j+1}] (dist={r['distance']:.4f}) [{r['metadata']['title']}]")
        print(f"      {r['text'][:100]}...")

---
## 5. RAG Query Pipeline

The complete RAG pipeline has three steps:
1. **Retrieve** -- Find relevant chunks from the vector store
2. **Augment** -- Inject retrieved chunks into the prompt as context
3. **Generate** -- Call the LLM with the augmented prompt

In [ ]:
RAG_SYSTEM_PROMPT = (
    "You are a helpful customer support agent for AcmeCorp. "
    "Answer questions based ONLY on the provided context. "
    "If the context does not contain the answer, say so clearly. "
    "Cite which document the information comes from."
)


def rag_query(question, n_context=3, verbose=True):
    """Complete RAG pipeline: retrieve, augment, generate."""

    # Step 1: Retrieve
    retrieved = search(question, n_results=n_context)

    if verbose:
        print("=== Step 1: Retrieved Chunks ===")
        for r in retrieved:
            print(f"  [{r['metadata']['title']}] distance={r['distance']:.4f}")

    # Step 2: Augment
    context_parts = []
    for r in retrieved:
        context_parts.append(f"[Source: {r['metadata']['title']}]\n{r['text']}")
    context = "\n\n---\n\n".join(context_parts)

    augmented_prompt = (
        f"Context from our documentation:\n\n{context}\n\n"
        f"---\n\nCustomer question: {question}"
    )

    if verbose:
        print(f"\n=== Step 2: Augmented Prompt ({len(augmented_prompt)} chars) ===")
        print(augmented_prompt[:300] + "...")

    # Step 3: Generate
    sources = list(set(r["metadata"]["title"] for r in retrieved))

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": RAG_SYSTEM_PROMPT},
                {"role": "user", "content": augmented_prompt}
            ],
            max_tokens=300,
            temperature=0.0
        )
        answer = response.choices[0].message.content
    except Exception as e:
        print(f"API call failed: {e}")
        # Mock response based on query content
        q_lower = question.lower()
        if "pricing" in q_lower or "cost" in q_lower:
            answer = (
                "Mock: Based on the Product Overview, DataFlow Pro pricing starts at "
                "$500/month for Starter, $2,000/month for Business, and custom pricing "
                "for Enterprise. The Pricing FAQ confirms a 14-day free trial with no "
                "credit card required, and annual billing gives a 20% discount."
            )
        elif "setup" in q_lower or "install" in q_lower:
            answer = (
                "Mock: According to the DataFlow Pro Setup Guide: 1) Install with "
                "`pip install dataflow-pro`, 2) Run `dfp init --project my-project`, "
                "3) Configure sources in config.yaml, 4) Define pipeline in pipeline.dfp, "
                "5) Deploy with `dfp deploy --env production`."
            )
        elif "soc" in q_lower or "security" in q_lower:
            answer = (
                "Mock: Yes, according to the Security and Compliance document, AcmeCorp "
                "is SOC 2 Type II certified and GDPR compliant. All data is encrypted "
                "at rest (AES-256) and in transit (TLS 1.3)."
            )
        else:
            answer = (
                f"Mock: Based on the retrieved documentation from {sources[0]}, "
                "the information relevant to your question is contained in our "
                "product documentation."
            )
        print("Using mock response for demonstration.")

    if verbose:
        print(f"\n=== Step 3: Generated Answer ===")

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "n_chunks": len(retrieved)
    }


# Demo: full RAG pipeline
result = rag_query("How much does DataFlow Pro cost and is there a free trial?")
print(result["answer"])

In [ ]:
# With vs. Without RAG comparison
comparison_question = "What is the pricing for DataFlow Pro and does AcmeCorp offer a free trial?"

# Without RAG
try:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Answer based only on your training data."},
            {"role": "user", "content": comparison_question}
        ],
        max_tokens=200
    )
    no_rag_answer = response.choices[0].message.content
except Exception as e:
    print(f"API call failed: {e}")
    no_rag_answer = (
        "Mock: I don't have specific information about AcmeCorp or DataFlow Pro "
        "in my training data. I'd recommend checking AcmeCorp's official "
        "website or contacting their sales team for pricing details."
    )
    print("Using mock response for demonstration.")

# With RAG
rag_result = rag_query(comparison_question, verbose=False)

print("=" * 70)
print("COMPARISON: With vs. Without RAG")
print("=" * 70)
print(f"\nQuestion: {comparison_question}")
print(f"\n--- WITHOUT RAG ---")
print(no_rag_answer)
print(f"\n--- WITH RAG ---")
print(rag_result["answer"])
print(f"\nSources: {', '.join(rag_result['sources'])}")
print("\n" + "=" * 70)
print("Key insight: Without RAG, the LLM cannot answer questions about")
print("proprietary information. RAG grounds the response in your documents.")

In [ ]:
# Test with multiple questions
test_questions = [
    "How do I set up DataFlow Pro?",
    "What security certifications does AcmeCorp have?",
    "Does InsightEngine support natural language queries?",
    "What is the capital of France?",  # Not in our knowledge base
]

print("=== RAG Pipeline: Multiple Queries ===")
for q in test_questions:
    print(f"\nQ: {q}")
    result = rag_query(q, verbose=False)
    print(f"A: {result['answer'][:200]}")
    print(f"   Sources: {', '.join(result['sources'])}")

---
## 6. Evaluation: Retrieval Quality

Evaluating RAG quality requires measuring **retrieval relevance** (did we find the right chunks?). We measure top-1 accuracy: does the top retrieved chunk come from the correct source document?

In [ ]:
# Retrieval accuracy test
test_cases = [
    {"query": "DataFlow Pro pricing", "expected_source": "product-overview"},
    {"query": "How to install DataFlow", "expected_source": "dataflow-setup"},
    {"query": "SOC 2 compliance", "expected_source": "security-compliance"},
    {"query": "InsightEngine natural language", "expected_source": "insightengine-features"},
    {"query": "Free trial available", "expected_source": "pricing-faq"},
]

print(f"{'Query':<35} {'Expected Source':<25} {'Top Result':<25} {'Match'}")
print("-" * 95)

hits = 0
for tc in test_cases:
    results = search(tc["query"], n_results=1)
    top_source = results[0]["metadata"]["source_doc"]
    match = top_source == tc["expected_source"]
    hits += int(match)
    status = "HIT" if match else "MISS"
    print(f"{tc['query']:<35} {tc['expected_source']:<25} {top_source:<25} {status}")

accuracy = hits / len(test_cases)
print(f"\nRetrieval Accuracy (top-1): {hits}/{len(test_cases)} = {accuracy:.0%}")

In [ ]:
# LLM-as-judge evaluation for answer quality
def evaluate_answer(question, answer, context):
    """Use LLM to evaluate RAG answer quality."""
    eval_prompt = (
        f"Evaluate this RAG system response.\n\n"
        f"Question: {question}\n"
        f"Context provided: {context[:500]}\n"
        f"Answer: {answer}\n\n"
        f"Rate on these criteria (1-5 each):\n"
        f"1. Relevance: Is the answer relevant to the question?\n"
        f"2. Faithfulness: Is the answer grounded in the provided context?\n"
        f"3. Completeness: Does it fully address the question?\n\n"
        f'Respond as JSON: {{"relevance": N, "faithfulness": N, "completeness": N}}'
    )

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": eval_prompt}],
            max_tokens=100,
            temperature=0.0,
            response_format={"type": "json_object"}
        )
        result = json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"API call failed: {e}")
        result = {"relevance": 4, "faithfulness": 5, "completeness": 4}
        print("Using mock response for demonstration.")

    return result


eval_questions = [
    "How much does DataFlow Pro cost?",
    "How do I deploy DataFlow Pro?",
    "What security certifications does AcmeCorp hold?",
]

print("=== Answer Quality Evaluation (LLM-as-Judge) ===")
print(f"{'Question':<42} {'Rel':>5} {'Faith':>6} {'Comp':>6} {'Avg':>6}")
print("-" * 70)

for q in eval_questions:
    rag_result = rag_query(q, verbose=False)
    retrieved = search(q, n_results=3)
    context_text = "\n".join([r["text"] for r in retrieved])
    scores = evaluate_answer(q, rag_result["answer"], context_text)
    vals = [scores.get(k, 0) for k in ["relevance", "faithfulness", "completeness"]]
    avg = sum(vals) / len(vals)
    print(f"{q[:40]:<42} {vals[0]:>5} {vals[1]:>6} {vals[2]:>6} {avg:>6.1f}")

print(f"\nScores: 1-5 (5 = best). Uses LLM-as-judge evaluation pattern.")

---
## Summary

This notebook built a complete RAG pipeline from scratch:

| Stage | What It Does | Key Decision |
|---|---|---|
| Chunking | Split documents into embeddable pieces | Chunk size and overlap |
| Embedding | Convert text to dense vectors | Model choice (cost vs quality) |
| Indexing | Store vectors in a vector database | Database choice (ChromaDB, Pinecone, etc.) |
| Retrieval | Find similar chunks for a query | Number of results (k) |
| Augmentation | Inject context into the prompt | Prompt template design |
| Generation | LLM produces grounded answer | Temperature, system prompt |
| Evaluation | Measure retrieval and answer quality | Metrics (accuracy, faithfulness) |

**Key takeaways:**
- RAG grounds LLM responses in your actual data, reducing hallucination
- Without RAG, the LLM cannot answer questions about proprietary information
- Chunk size significantly impacts retrieval quality
- Always evaluate both retrieval relevance and answer faithfulness

**Next:** In notebook 04, we apply LLMs to structured document processing and data extraction.